# Notebook 4: Model Training
## Project: Crop Stress and Irrigation Priority Decision Support System
**Objective:** Split the data, scale features where appropriate, train candidate Machine Learning algorithms for classification and regression, and generate initial test predictions.

### Step 1: Import Required Libraries
We restrict imports strictly to standard Scikit-learn, NumPy, and Pandas tools.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Classification Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Regression Algorithms
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

### Step 2: Load the Clean Preprocessed Dataset

In [2]:
try:
    df = pd.read_csv("data/processed/krishijal_ml_clean.csv")
    print("Loaded data/processed/krishijal_ml_clean.csv")
except:
    try:
        df = pd.read_csv("krishijal_ml_clean.csv")
        print("Loaded krishijal_ml_clean.csv")
    except:
        df = pd.read_csv("data/processed/krishijal_master_dataset.csv")
        print("Loaded data/processed/krishijal_master_dataset.csv")

Loaded krishijal_ml_clean.csv


### Step 3: Define Features and Target Variables
We separate input features ($X$) from the classification target ($y_{class}$) and regression target ($y_{reg}$).

In [3]:
feature_cols = [
    "cwsi", "effective_soil_wetness", "water_deficit", "ndvi",
    "et0", "etc", "temperature", "rainfall", "humidity"
]

X = df[feature_cols]
y_class = df["stress_level_code"]
y_reg = df["irrigation_priority_index"]

print("Feature Matrix Shape:", X.shape)
print("Classification Target Count:", len(y_class))
print("Regression Target Count:", len(y_reg))

Feature Matrix Shape: (6654, 9)
Classification Target Count: 6654
Regression Target Count: 6654


### Step 4: Perform Train-Test Split (Prevent Data Leakage)
We split 80% for training and 20% for testing. For classification, we use `stratify=y_class` to ensure proportional representation of all four stress severity classes.

In [4]:
X_train, X_test, y_train_class, y_test_class = train_test_split(
    X, y_class, test_size=0.20, random_state=42, stratify=y_class
)

_, _, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.20, random_state=42
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

Training Samples: 5323
Testing Samples: 1331


### Step 5: Feature Scaling (StandardScaler)
StandardScaler transforms features to zero mean and unit variance ($z = (x - \mu)/\sigma$).
**Crucial ML Practice:** We fit the scaler strictly on `X_train` and then apply it to `X_test` to prevent data leakage.

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete. X_train_scaled mean:", np.round(X_train_scaled.mean(), 4))

Feature scaling complete. X_train_scaled mean: 0.0


### Step 6: Train Classification Models
We train three suitable classification algorithms:
1. **Logistic Regression:** Linear baseline using scaled features.
2. **Decision Tree Classifier:** Non-linear rule-based tree model.
3. **Random Forest Classifier:** Bagging ensemble of decision trees.

In [6]:
print("Training Logistic Regression...")
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_scaled, y_train_class)

print("Training Decision Tree Classifier...")
model_dt_class = DecisionTreeClassifier(max_depth=5, random_state=42)
model_dt_class.fit(X_train, y_train_class)

print("Training Random Forest Classifier (Initial Baseline)...")
model_rf_class = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=42)
model_rf_class.fit(X_train, y_train_class)

print("All classification models successfully trained.")

Training Logistic Regression...
Training Decision Tree Classifier...
Training Random Forest Classifier (Initial Baseline)...
All classification models successfully trained.


### Step 7: Generate Classification Predictions
We generate predictions on the unseen test dataset.

In [7]:
pred_lr = model_lr.predict(X_test_scaled)
pred_dt_class = model_dt_class.predict(X_test)
pred_rf_class = model_rf_class.predict(X_test)

print("Sample predictions from Random Forest Classifier (first 10):", pred_rf_class[:10])
print("Actual test labels (first 10):                     ", y_test_class.values[:10])

Sample predictions from Random Forest Classifier (first 10): [0 2 0 0 0 0 0 0 2 2]
Actual test labels (first 10):                      [0 2 0 0 0 0 0 0 2 2]


### Step 8: Train Regression Models
We train three suitable regression algorithms to predict `irrigation_priority_index`:
1. **Linear Regression:** Parametric linear baseline.
2. **Decision Tree Regressor:** Non-linear decision tree regressor.
3. **Random Forest Regressor:** Non-linear ensemble regressor.

In [8]:
print("Training Linear Regression...")
model_lin = LinearRegression()
model_lin.fit(X_train_scaled, y_train_reg)

print("Training Decision Tree Regressor...")
model_dt_reg = DecisionTreeRegressor(max_depth=5, random_state=42)
model_dt_reg.fit(X_train, y_train_reg)

print("Training Random Forest Regressor (Initial Baseline)...")
model_rf_reg = RandomForestRegressor(n_estimators=20, max_depth=5, random_state=42)
model_rf_reg.fit(X_train, y_train_reg)

print("All regression models successfully trained.")

Training Linear Regression...
Training Decision Tree Regressor...
Training Random Forest Regressor (Initial Baseline)...


All regression models successfully trained.


### Step 9: Generate Regression Predictions
We generate predictions on the unseen test dataset.

In [9]:
pred_lin = model_lin.predict(X_test_scaled)
pred_dt_reg = model_dt_reg.predict(X_test)
pred_rf_reg = model_rf_reg.predict(X_test)

print("Sample predicted Irrigation Priority Index (first 5):", np.round(pred_rf_reg[:5], 2))
print("Actual Irrigation Priority Index (first 5):          ", np.round(y_test_reg.values[:5], 2))

Sample predicted Irrigation Priority Index (first 5): [21.24 21.01 20.43 21.5  19.82]
Actual Irrigation Priority Index (first 5):           [23.4 36.3  9.6  7.5 10.4]


### Step 10: Summary of Model Training
- Clean train/test split completed (5,323 train / 1,331 test).
- Scaler fitted only on training data (no leakage).
- 3 classification models and 3 regression models successfully trained and predicted.
- In the next notebook (`05_model_evaluation.ipynb`), we evaluate and tune hyperparameters.